In [1]:
pip install transformers datasets torch accelerate evaluate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import torch

print(torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

2.9.1+cpu
CUDA Available: False


In [5]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.9.1+cpu
False


In [6]:
import pandas as pd
import re

# Load datasets
fake = pd.read_csv("../data/Fake.csv")
true = pd.read_csv("../data/True.csv")

# Labels
fake["label"] = 1
true["label"] = 0

# Merge
df = pd.concat([fake, true], ignore_index=True)

# Remove duplicates
df = df.drop_duplicates()

# Create content column
df["content"] = df["title"] + " " + df["text"]

# Cleaning function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_content"] = df["content"].apply(clean_text)

print(df.shape)

(44689, 7)


In [7]:
import os

os.makedirs("src", exist_ok=True)

print("✅ src folder ready")

✅ src folder ready


In [11]:
fake_samples = df[df["label"] == 1].head(3)
real_samples = df[df["label"] == 0].head(3)

samples = pd.concat([fake_samples, real_samples])

with open("src/test_samples.txt", "w", encoding="utf-8") as f:
    for idx, row in samples.iterrows():

        news_type = "FAKE NEWS" if row["label"] == 1 else "REAL NEWS"

        f.write("=" * 80 + "\n")
        f.write(f"{news_type}\n")
        f.write("=" * 80 + "\n\n")

        f.write("ORIGINAL:\n")
        f.write(row["content"] + "\n\n")

        f.write("CLEANED:\n")
        f.write(row["clean_content"] + "\n\n")

print("✅ test_samples.txt updated with 3 Fake + 3 Real samples")

✅ test_samples.txt updated with 3 Fake + 3 Real samples


In [12]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFile size:")
print(os.path.getsize("src/test_samples.txt"))

Current working directory:
x:\fake-news-prediction\notebooks

File size:
41022


In [13]:
with open("src/test_samples.txt", "r", encoding="utf-8") as f:
    print(f.read()[:1000])  # first 1000 characters

FAKE NEWS

ORIGINAL:
 Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year,  President Angry Pants tweeted.  2018 will be a great year for America! As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year. 2018 will be a great year 


In [5]:
from sklearn.model_selection import train_test_split

# Use 10,000 samples for DistilBERT
df_small = df.sample(n=10000, random_state=42)

X_small = df_small["clean_content"]
y_small = df_small["label"]

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_small,
    y_small,
    test_size=0.2,
    random_state=42,
    stratify=y_small
)

print("Train:", len(X_train_b))
print("Test:", len(X_test_b))

Train: 8000
Test: 2000


In [6]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

print("Transformers loaded successfully")

Transformers loaded successfully


In [7]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

print("Tokenizer loaded")

Tokenizer loaded


In [8]:
print("Train:", len(X_train_b))
print("Test:", len(X_test_b))

Train: 8000
Test: 2000


In [9]:
DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

DistilBertTokenizerFast(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [10]:
import pandas as pd
from datasets import Dataset

train_df = pd.DataFrame({
    "text": X_train_b,
    "label": y_train_b
})

test_df = pd.DataFrame({
    "text": X_test_b,
    "label": y_test_b
})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 8000
})
Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 2000
})


In [11]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization completed")

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenization completed


In [12]:
print(tokenized_train)

Dataset({
    features: ['text', 'label', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 8000
})


In [13]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print("Model loaded successfully")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully


In [14]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    logging_steps=100,
    load_best_model_at_end=True,
)

In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

In [17]:
trainer.train()

c:\Users\docto\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.008900,0.026588,0.996000


TrainOutput(global_step=2000, training_loss=0.028268903163494542, metrics={'train_runtime': 10132.6257, 'train_samples_per_second': 0.79, 'train_steps_per_second': 0.197, 'total_flos': 529869594624000.0, 'train_loss': 0.028268903163494542, 'epoch': 1.0})

In [18]:
trainer.save_model("../models/distilbert_fake_news")

tokenizer.save_pretrained("../models/distilbert_fake_news")

('../models/distilbert_fake_news\\tokenizer_config.json',
 '../models/distilbert_fake_news\\special_tokens_map.json',
 '../models/distilbert_fake_news\\vocab.txt',
 '../models/distilbert_fake_news\\added_tokens.json',
 '../models/distilbert_fake_news\\tokenizer.json')

In [19]:
predictions = trainer.predict(tokenized_test)

preds = predictions.predictions.argmax(axis=-1)

from sklearn.metrics import classification_report

print(classification_report(y_test_b, preds))

c:\Users\docto\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


              precision    recall  f1-score   support

           0       1.00      0.99      1.00       939
           1       0.99      1.00      1.00      1061

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [20]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_b, preds)
print(cm)

[[ 932    7]
 [   1 1060]]


Logistic Regression Accuracy: 98.96%

DistilBERT Accuracy: 99.60%

Improvement:
DistilBERT achieved better performance by understanding contextual meaning rather than only word frequency patterns.

# Results

Logistic Regression Accuracy: 98.96%

DistilBERT Accuracy: 99.60%

Conclusion:
DistilBERT outperformed the baseline Logistic Regression model by capturing contextual information and semantic meaning.